```
“불균형 데이터(imbalanced data)” 다루기
데이터의 클래스(라벨) 비율이 불균형할 때 모델을 제대로 학습시키고 공정하게 평가하는 방법
```
#### 클래스 가중치 사용
#### 리셈플링 기법
#### 적절한 평가지표

In [2]:
# 불균형 데이터 생성 (1:9) 악성 0 / 양성 1

# 유방암 데이터 불러오기
from sklearn.datasets import load_breast_cancer
import numpy as np
np.random.seed(42)

data = load_breast_cancer()
X,y = data.data, data.target

In [3]:
# 1과 0 (악성과 양성)의 비율 조회
np.unique(y,return_counts=True)
print(f'악성과 양성 비율 : {np.unique(y,return_counts=True)}')
# 악성 : 121개 , 양성 : 357개

# 0인 데이터의 인덱스 조회
np.where(y == 0) # index 변환
print(f'악성데이터 인덱스 목록 : {np.where(y == 0)}')

# 1인 데이터의 인덱스 조회
np.where(y == 1)
print(f'양성데이터 인덱스 목록 : {np.where(y == 1)}')

악성과 양성 비율 : (array([0, 1]), array([212, 357]))
악성데이터 인덱스 목록 : (array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  22,  23,  24,  25,  26,  27,  28,
        29,  30,  31,  32,  33,  34,  35,  36,  38,  39,  40,  41,  42,
        43,  44,  45,  47,  53,  54,  56,  57,  62,  64,  65,  70,  72,
        73,  75,  77,  78,  82,  83,  85,  86,  87,  91,  94,  95,  99,
       100, 105, 108, 117, 118, 119, 121, 122, 126, 127, 129, 131, 132,
       134, 135, 138, 141, 146, 156, 161, 162, 164, 167, 168, 171, 172,
       177, 180, 181, 182, 184, 186, 190, 193, 194, 196, 197, 198, 199,
       201, 202, 203, 205, 207, 210, 212, 213, 214, 215, 218, 219, 223,
       229, 230, 233, 236, 237, 239, 244, 250, 252, 253, 254, 255, 256,
       257, 258, 259, 260, 261, 262, 263, 264, 265, 272, 274, 277, 280,
       282, 283, 297, 300, 302, 317, 321, 323, 328, 329, 330, 335, 337,
       339, 343, 351, 352, 353, 365, 366, 368, 369, 370, 372, 373, 379,
 

In [3]:
m_index = np.where(y == 0)[0]
b_index = np.where(y == 1)[0]

# 악성 데이터가 많으니 일부만, 양성은 더 많이 사용할수있도록 설정
# 악성의 30% 사용, 양성은 전체 1.5 : 8.5
# 악성을 소수 클래스로 생성
size_30 = int(len(m_index)*0.2)
selected_m_index = np.random.choice(m_index,size=size_30,replace=False)
selected_b_index = b_index
len(selected_m_index), len(selected_b_index)
concatenate_selected_index = np.concatenate([selected_m_index, selected_b_index])
np.random.shuffle(concatenate_selected_index)

X_imb = X[concatenate_selected_index]
y_imb = y[concatenate_selected_index]

# 클래스 분포 확인
unique, counts = np.unique(y_imb,return_counts=True)
print('클래스 분포 :')
for label, count in zip(unique,counts):
    percentage = count / len(y_imb) * 100
    print(f'클래스 {label} : {count} 개 : {percentage:1f}%')

클래스 분포 :
클래스 0 : 42 개 : 10.526316%
클래스 1 : 357 개 : 89.473684%


In [4]:
# 불균형인 상태로 진행
# 스케일링 정규화 StandardScaler
# LoogisticRegression
# pipe
# 평가는 class report

In [5]:
# 학생풀이
# 학습 데이터 / 테스트 데이터 분류하기
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X_imb,y_imb,test_size=0.2,random_state=42)

# 스케일링 : 정규화
from sklearn.preprocessing import StandardScaler
st = StandardScaler()
X_train_scaled = st.fit_transform(X_train)

In [6]:
# 로지스틱 회귀모델 생성
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression()
clf.fit(X_train_scaled,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [7]:
# 파이프라인 생성
from sklearn.pipeline import Pipeline
pipe = Pipeline([
    ('scaler',StandardScaler()),
    ('clf',LogisticRegression(random_state=42,max_iter=1000))
])
pipe.fit(X_train,y_train)

,steps,"[('scaler', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0


In [ ]:

# class report 로 모델 평가하기
from sklearn.metrics import classification_report
# 모델 예측 수행
y_pred = pipe.predict(X_test)

# 리포트 출력
print( classification_report(y_test, y_pred,target_names=['악성(0)','양성(1)'] )  )

# 실행을 할때마다 결과값이 달라짐 -> 넘파이에서 사용되는 랜덤값에서 나오는 변화 -> 특정 시드값으로 고정하기 np.random.seed(42)

              precision    recall  f1-score   support

       악성(0)       1.00      0.75      0.86         8
       양성(1)       0.97      1.00      0.99        72

    accuracy                           0.97        80
   macro avg       0.99      0.88      0.92        80
weighted avg       0.98      0.97      0.97        80



In [9]:
# 강사풀이
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# x_train,x_test,y_train,y_test = train_test_split(X_imb, y_imb,stratify=y_imb, test_size=0.2,random_state=42)
print('1. 기본 모델 (불균형 무시)')
pipe = Pipeline([
    ('scaler',StandardScaler()),
    ('clf', LogisticRegression(random_state=42,max_iter=1000)) # ^ 로지스틱 회귀 설정을 이렇게 한 이유 ^
])
pipe.fit(X_train, y_train)
print( classification_report(y_test, pipe.predict(X_test),target_names=['악성(0)','양성(1)'] )  )

1. 기본 모델 (불균형 무시)
              precision    recall  f1-score   support

       악성(0)       1.00      0.75      0.86         8
       양성(1)       0.97      1.00      0.99        72

    accuracy                           0.97        80
   macro avg       0.99      0.88      0.92        80
weighted avg       0.98      0.97      0.97        80



In [10]:
print('불균형 해결 : 클래스 가중치 사용')
# 기존 파이프라인의 clf이름의 객체의 파라미터를 조정
from copy import deepcopy
pipe_weight = deepcopy(pipe)
pipe_weight = pipe.set_params(clf__class_weight = 'balanced') # class_weight={0: 5, 1: 1} 이런식으로도 직접 조정가능하다 : 데이터 0에 5배 가중치
pipe_weight.fit(X_train,y_train)

# 기존 파이프라인에서 clf이름의 객체의 파라미터를 조정한 파이프라인(pipe_weight)으로 모델 예측
y_pred_weight = pipe_weight.predict(X_test) 
print(classification_report(y_test,y_pred_weight))

print("class_weight = 'balanced'로 자동 지정된 가중치가 얼마인지 계산")
#


불균형 해결 : 클래스 가중치 사용
              precision    recall  f1-score   support

           0       1.00      0.88      0.93         8
           1       0.99      1.00      0.99        72

    accuracy                           0.99        80
   macro avg       0.99      0.94      0.96        80
weighted avg       0.99      0.99      0.99        80

class_weight = 'balanced'로 자동 지정된 가중치가 얼마인지 계산


In [11]:
print(' 불균형 해결 : RandomForest(균형모드)')
from sklearn.ensemble import RandomForestClassifier

# 랜덤포레스트를 적용한 파이프라인 생성
pipe_rf = Pipeline([
    ('scaler',StandardScaler()),
    ('clf',RandomForestClassifier(class_weight='balanced',random_state=42))
])
pipe_rf.fit(X_train,y_train)

# 모델 예측 수행
y_pred_rf_weight = pipe_rf.predict(X_test)

# 리포트 출력
print(classification_report(y_test,y_pred_rf_weight))

 불균형 해결 : RandomForest(균형모드)
              precision    recall  f1-score   support

           0       1.00      0.88      0.93         8
           1       0.99      1.00      0.99        72

    accuracy                           0.99        80
   macro avg       0.99      0.94      0.96        80
weighted avg       0.99      0.99      0.99        80



In [ ]:
%pip install imbalanced-learn

In [ ]:
print(' 불균형 해결 : 데이터 오버 / 언더 샘플링')

# 데이터 오버 샘플링
from imblearn.over_sampling import RandomOverSampler, SMOTE
from collections import Counter

# 클래스 분포 확인
unique, counts = np.unique(y_imb,return_counts=True)
print('클래스 분포 :')
for label, count in zip(unique,counts):
    percentage = count / len(y_imb) * 100
    print(f'클래스 {label} : {count} 개 : {percentage:1f}%')

 불균형 해결 : 데이터 오버 / 언더 샘플링


In [ ]:
# 데이터 언더 샘플링
from imblearn.under_sampling import RandomUnderSampler
